# Session 13 · Case study: leaderboard round L3 with a TF-IDF classifier

**Block 2 practice.** Train a TF-IDF + linear classifier on the 50,000-review sample, check it with per-class metrics, and write a leaderboard submission (`review_id,label`).

Reference values on the public leaderboard (macro-F1): always "pos" 0.26 · simple text features 0.49 · **TF-IDF on the 50k sample 0.69** · TF-IDF on the full training set 0.71.

Theory: [02-tfidf-and-text-classification.md](../theory/02-tfidf-and-text-classification.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
texts = reviews["title"] + " " + reviews["text"]          # one document = title + text
y = reviews["label"]
print(reviews.shape, y.value_counts(normalize=True).round(3).to_dict())

## 1. Validation split and a first model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

X_tr, X_va, y_tr, y_va = train_test_split(texts, y, test_size=0.2, stratify=y, random_state=0)

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
    LogisticRegression(C=4, class_weight="balanced", max_iter=2000),
)
model.fit(X_tr, y_tr)
pred = model.predict(X_va)
print("macro-F1:", round(f1_score(y_va, pred, average="macro"), 3))
print(classification_report(y_va, pred, digits=2))

**Question.** Which class has the lowest F1? Compare precision and recall for that class: does the model predict it too often or too rarely?

## 2. A small comparison of settings

Each fit takes a few seconds. Keep the grid small and decide on the validation split, never on the leaderboard.

In [ ]:
import time

from sklearn.svm import LinearSVC

candidates = {
    "unigrams, C=4": dict(ngram_range=(1, 1), clf=LogisticRegression(C=4, class_weight="balanced", max_iter=2000)),
    "1-2-grams, C=1": dict(ngram_range=(1, 2), clf=LogisticRegression(C=1, class_weight="balanced", max_iter=2000)),
    "1-2-grams, C=4": dict(ngram_range=(1, 2), clf=LogisticRegression(C=4, class_weight="balanced", max_iter=2000)),
    "1-2-grams, C=4, unbalanced": dict(ngram_range=(1, 2), clf=LogisticRegression(C=4, max_iter=2000)),
    "1-2-grams, linear SVM": dict(ngram_range=(1, 2), clf=LinearSVC(C=0.5, class_weight="balanced")),
}
rows = []
for name, c in candidates.items():
    pipe = make_pipeline(TfidfVectorizer(ngram_range=c["ngram_range"], min_df=3, sublinear_tf=True), c["clf"])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    p = pipe.predict(X_va)
    rows.append({"model": name, "macro_F1": f1_score(y_va, p, average="macro"),
                 "F1_neu": f1_score(y_va, p, labels=["neu"], average="macro"),
                 "seconds": time.perf_counter() - t0})
pd.DataFrame(rows).round(3).sort_values("macro_F1", ascending=False)

## 3. Refit on all sample reviews and predict the test set

In [ ]:
test = pd.read_parquet(DATA / "test.parquet")
test_texts = test["title"] + " " + test["text"]

final = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
    LogisticRegression(C=4, class_weight="balanced", max_iter=2000),
)
final.fit(texts, y)                                   # all 50,000 reviews
submission = pd.DataFrame({"review_id": test["review_id"], "label": final.predict(test_texts)})
print(submission.shape)
print(submission["label"].value_counts(normalize=True).round(3))

Checks before submitting: one row per test review, no missing ids, only the three labels.

In [ ]:
assert len(submission) == len(test) and submission["review_id"].is_unique
assert set(submission["label"]) <= {"neg", "neu", "pos"}
path = Path("submission-L3-tfidf.csv")
submission.to_csv(path, index=False)
print("written:", path.resolve())

## 4. Score

Upload the file to the course leaderboard, or score it locally where the solution file is available (lecturer):

```bash
uv run --with pandas --with scikit-learn python case-study/score.py sessions/13-classical-nlp/workbooks/submission-L3-tfidf.csv
```

The next cell does the same if `case-study/data/instructor/solution.csv` exists.

In [ ]:
import sys

solution = DATA / "instructor" / "solution.csv"
if solution.exists():
    sys.path.insert(0, str(ROOT / "case-study"))
    from score import score
    for k, v in score(submission, pd.read_csv(solution)).items():
        print(f"{k:22s} {v:.4f}")
else:
    print("No solution file: upload the CSV to the leaderboard instead.")

## Your turn

1. Why is the public leaderboard score (about 0.69) lower than the validation score (about 0.74)? Write down two hypotheses; Block 3 tests one of them.
2. Train the same pipeline on the full training set (`train.parquet`, 434,373 reviews; about one minute) and submit again. The reference value is 0.71.
3. Record your submission in the team log: date, model, settings, validation macro-F1, leaderboard macro-F1.

In [ ]:
# your code here